# TN2211 Session 3

In this session, you will start by making observations by hand using the DMM, writing your results down on a piece of paper and analysing it there by hand. You are free to use python here as a "calculator" if you like, it is then a handy calculator with a history built in.

In step 5, you will make a plot based on your hand recorded data, and there is some code here to help you out with that.

If you venture into Level 2 and want to explore the IV curve of a diode, you will need to record scope traces, and for that there is some code included below. 

## Instruments

In [ ]:
import sys
sys.path.append("../drivers/")
from tn2211_drivers import *
import glob
import matplotlib.pyplot as plt

In [ ]:
import pyvisa
rm = pyvisa.ResourceManager()
rm.list_resources()

In [ ]:
scope = Scope("SDS")

In [ ]:
scope.get_screenshot()

In [ ]:
t,v = scope.get_trace(1)

In [ ]:
gen = Generator("SDG")

In [ ]:
gen.set_frequency(1,10e3)

In [ ]:
gen.get_screenshot()

In [ ]:
help(gen)

## Calculator 

It is always handy to have some code cells to do quick calculations for you. 

In [ ]:
R = 100e3 # Ohms
C = 10e-9 # Farads

tau = R*C

print("tau %e seconds" % tau)

In [ ]:
t_90_10 = 3.3e-4 # seconds, fill in your observation from the scope

tau = t_90_10 / 2.2

print("t_90_10 is %e" % t_90_10)
print("tau is %e seconds" % tau)

In [ ]:
# add cells as needed for your calculations

In [ ]:
# more calculations

## Plots of $\tau_{measured}$ and $\tau_{predicted}$

Some code to help you make the plots

In [ ]:
R = # your fixed r value in ohms
C = np.array([xxx, xxx, xxx, xxx]) # your C values in farads

tau_predicted = R*C

t_10_90_measured = np.zeros(4)
t_10_90_measured[0] =  xxx # seconds
t_10_90_measured[1] =  xxx # seconds
t_10_90_measured[2] =  xxx # seconds
t_10_90_measured[3] =  xxx # seconds

tau_meas = t_10_90_measured / 2.2

plt.title("R = %e ohms", R) # handy for documenting
plt.plot(C, tau_meas, 'o', label="...")
plt.plot(C, tau_predicted, '-', label="...")
plt.legend()
# add labels etc


# Fitting a transient

For exmaple, for an RC circuit, the voltage for a step response of height $V_0$ will give, for $t>0$, given by: 

$$ 
V(t) = V_0 (1 - e^{-t/\tau})
$$

In practice, your real data may have an offset voltage, and maybe also the transient you are trying to fit is not exactly at $t=0$ of your trace. In this case, you will want to add two more parameters: 

$$
V(t) = V_0 (1-e^{-(t-t_0)/\tau}) + V_{off}
$$

Note that this formula applies only for $t>0$ and for an single impulse. That is not what is recorded on your oscilloscope trace: it includes voltage measurements before the step, and also the steps are not infinitey long. The first problem can be solved by fitting only a subset of the data, and the second can be solved by making sure the period of your square pulse you use is very long compared to the decay time. 

To deal with the first problem, you slide your data to include only part of the data for which you think the model approximately represents your experimental data. 

In [ ]:
t,v = scope.get_trace(1)

i1 = 100 # where to start slicing the data, the point at which you think the model starts to apply
i2 = 200 # the last bit of the data to fit

def v_model(t, V0, t0, tau, Voff):
    return V0*(1-np.exp((t-t0)/tau)) + Voff

t_fit = t[i1:i2]
v_fit = v[i1:i2]

# Initial guess

V0 = 1 # V
t0 = 0 # seconds
tau = 1 # seconds
Voff = 0 # V

plt.plot(t,v, label="Full trace") # the full trace
plt.plot(t_fit, v_fit, label="Fitted region") # the data we slice out to fit
plt.plot(t_fit, v_model(t_fit, V0, t0, tau, Voff), label="Iniital guess") 

In [21]:
# Now run curve fit (scipy is good, lmfit is even better), 
# replot with fitted data, look at fit values, errors, and covariance

# your code

In our simple funciton above, we already have 4 free fit parmaters! Free fit parameters are bad in general because you can often end up in a "local minimum" where your function fits the data well but might be (far?) off from the actual underlying physical parameters of your circuit.

One way to solve this is ot use **independent** measurements to separately fix parameters and then rewrite a different free parameters. (In [scipy curve_fit](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.curve_fit.html), you have to write a new fucntion, while the [lmfit library](https://lmfit.github.io/lmfit-py/model.html) allows you to resuse the same function and the specify which parameters of your fit should be "free" and which should be fixed.)

If your square pulse period is long enough, you can extract the offset voltage by slicing out points that you know are after the transient has settled and then using np.average(). Similarly, if you want to get an accurate estimate of `t0`, you could enable a second channel on your scope and use a jumper on your PCB to *measure* the square pulse applied to your circuit and download it to python to determine accurately the switching time. 

In [22]:
# Try a fit where you independently fix Voff and t0, document how you get these values!

# your code